# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from itertools import product
from sklearn.ensemble import BaggingClassifier
import warnings
from sklearn.exceptions import UndefinedMetricWarning
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
import joblib


## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [2]:
# Preprocessing
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
df['dayofweek'] = pd.read_csv('../data/dayofweek.csv')['dayofweek']

# Разделение на признаки и целевую переменную
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

# Первый сплит: train/test (80/20)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=21,
    stratify=y
)

# Второй сплит: train/valid (из train_full)
# По умолчанию split делит 75/25, но можно указать test_size=0.2 для 80/20
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    random_state=21,
    stratify=y_train_full
)

# Проверка размеров
print(f'Full dataset: {len(df)} samples')
print(f'Train: {len(X_train)} samples ({len(X_train)/len(df)*100:.1f}%)')
print(f'Valid: {len(X_valid)} samples ({len(X_valid)/len(df)*100:.1f}%)')
print(f'Test: {len(X_test)} samples ({len(X_test)/len(df)*100:.1f}%)')

# Проверка распределения классов
print('\nClass distribution in train:')
print(y_train.value_counts().sort_index())
print('\nClass distribution in valid:')
print(y_valid.value_counts().sort_index())
print('\nClass distribution in test:')
print(y_test.value_counts().sort_index())

Full dataset: 1686 samples
Train: 1078 samples (63.9%)
Valid: 270 samples (16.0%)
Test: 338 samples (20.0%)

Class distribution in train:
0     87
1    175
2     95
3    253
4     66
5    174
6    228
Name: dayofweek, dtype: int64

Class distribution in valid:
0    22
1    44
2    24
3    63
4    17
5    43
6    57
Name: dayofweek, dtype: int64

Class distribution in test:
0    27
1    55
2    30
3    80
4    21
5    54
6    71
Name: dayofweek, dtype: int64


## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [3]:
svm_model = SVC(
    C=10,
    kernel='rbf',
    gamma='auto',
    class_weight=None,
    probability=True,
    random_state=21
)
svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_valid)

print(f'accuracy is {accuracy_score(y_valid, y_pred_svm):.5f}')
print(f'precision is {precision_score(y_valid, y_pred_svm, average="weighted"):.5f}')
print(f'recall is {recall_score(y_valid, y_pred_svm, average="weighted"):.5f}')

accuracy is 0.87778
precision is 0.88162
recall is 0.87778


In [4]:
dt_model = DecisionTreeClassifier(
    max_depth=21,
    class_weight='balanced',
    criterion='gini',
    random_state=21
)
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_valid)

print(f'accuracy is {accuracy_score(y_valid, y_pred_dt):.5f}')
print(f'precision is {precision_score(y_valid, y_pred_dt, average="weighted"):.5f}')
print(f'recall is {recall_score(y_valid, y_pred_dt, average="weighted"):.5f}')

accuracy is 0.86667
precision is 0.87170
recall is 0.86667


In [5]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=24,
    class_weight='balanced',
    criterion='entropy',
    random_state=21
)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_valid)

print(f'accuracy is {accuracy_score(y_valid, y_pred_rf):.5f}')
print(f'precision is {precision_score(y_valid, y_pred_rf, average="weighted"):.5f}')
print(f'recall is {recall_score(y_valid, y_pred_rf, average="weighted"):.5f}')

accuracy is 0.89630
precision is 0.89698
recall is 0.89630


## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [6]:
results = []

for voting_type in ['soft', 'hard']:
    for w in product(range(1,6), repeat=3):
        voting_clf_w = VotingClassifier(
            estimators=[
                ('svm', svm_model),
                ('dt', dt_model),
                ('rf', rf_model)
            ],
            voting=voting_type,
            weights=list(w)
        )
        voting_clf_w.fit(X_train, y_train)
        y_pred_w = voting_clf_w.predict(X_valid)
        
        results.append({
            'voting': voting_type,
            'weights': w,
            'accuracy': accuracy_score(y_valid, y_pred_w),
            'precision': precision_score(y_valid, y_pred_w, average='weighted'),
            'recall': recall_score(y_valid, y_pred_w, average='weighted')
        })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(['accuracy', 'precision'], ascending=[False, False])
print(results_df.head(10).to_string(index=False))

voting    weights  accuracy  precision    recall
  soft  (4, 1, 4)  0.911111   0.912881  0.911111
  soft  (4, 1, 5)  0.911111   0.911443  0.911111
  soft  (5, 1, 1)  0.907407   0.911495  0.907407
  soft  (5, 1, 2)  0.907407   0.911495  0.907407
  soft  (4, 1, 3)  0.907407   0.910989  0.907407
  soft  (4, 1, 1)  0.907407   0.910258  0.907407
  soft  (4, 1, 2)  0.907407   0.910258  0.907407
  soft  (3, 2, 4)  0.907407   0.910121  0.907407
  soft  (5, 3, 3)  0.907407   0.909704  0.907407
  soft  (5, 1, 5)  0.907407   0.909564  0.907407


In [7]:
# Выбираем лучшие параметры
best_row = results_df.iloc[0]
best_voting = best_row['voting']
best_weights = best_row['weights']

print(f'Best voting: {best_voting}')
print(f'Best weights: {best_weights}')
print(f'Best valid accuracy: {best_row["accuracy"]:.5f}')

# Обучаем финальную модель
voting_best = VotingClassifier(
    estimators=[
        ('svm', svm_model),
        ('dt', dt_model),
        ('rf', rf_model)
    ],
    voting=best_voting,
    weights=list(map(int, best_weights))
)
voting_best.fit(X_train, y_train)

# Оценка на тесте
y_pred_test = voting_best.predict(X_test)

print(f'\naccuracy is {accuracy_score(y_test, y_pred_test):.5f}')
print(f'precision is {precision_score(y_test, y_pred_test, average="weighted"):.5f}')
print(f'recall is {recall_score(y_test, y_pred_test, average="weighted"):.5f}')

Best voting: soft
Best weights: (4, 1, 4)
Best valid accuracy: 0.91111

accuracy is 0.90533
precision is 0.90881
recall is 0.90533


## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [8]:
warnings.filterwarnings('ignore', category=UndefinedMetricWarning)

# Базовая модель SVM с лучшими параметрами
base_svm = SVC(
    C=10,
    kernel='rbf',
    gamma='auto',
    class_weight=None,
    probability=True,
    random_state=21
)

# Перебор параметров
n_estimators_list = [10, 20, 30, 47, 48, 49, 50, 51, 52, 53, 70, 100]
max_samples_list = [0.4, 0.5, 0.6, 0.7, 0.8 , 0.9, 1.0]
max_features_list = [0.4, 0.5, 0.6, 0.7, 0.8 , 0.9, 1.0]

results = []

for n_est in n_estimators_list:
    for max_samp in max_samples_list:
        for max_feat in max_features_list:
            bagging_clf = BaggingClassifier(
                base_estimator=base_svm,
                n_estimators=n_est,
                max_samples=max_samp,
                max_features=max_feat,
                bootstrap=True,
                bootstrap_features=False,
                n_jobs=-1,
                random_state=21
            )
            bagging_clf.fit(X_train, y_train)
            y_pred = bagging_clf.predict(X_valid)
            
            results.append({
                'n_estimators': n_est,
                'max_samples': max_samp,
                'max_features': max_feat,
                'accuracy': accuracy_score(y_valid, y_pred),
                'precision': precision_score(y_valid, y_pred, average='weighted'),
                'recall': recall_score(y_valid, y_pred, average='weighted')
            })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(['accuracy', 'precision'], ascending=[False, False])
print(results_df.head(10).to_string(index=False))

 n_estimators  max_samples  max_features  accuracy  precision    recall
           30          1.0           1.0  0.888889   0.897182  0.888889
           10          1.0           1.0  0.885185   0.894269  0.885185
           52          1.0           1.0  0.885185   0.893961  0.885185
           53          1.0           1.0  0.885185   0.893961  0.885185
           70          0.9           1.0  0.885185   0.893961  0.885185
           70          1.0           1.0  0.885185   0.893961  0.885185
          100          1.0           1.0  0.885185   0.893961  0.885185
           20          1.0           1.0  0.885185   0.892576  0.885185
           50          1.0           0.9  0.885185   0.891848  0.885185
           51          1.0           0.9  0.885185   0.891848  0.885185


In [9]:
# Выбираем лучшие параметры
best_row = results_df.iloc[0]
best_n_est = int(best_row['n_estimators'])
best_max_samp = best_row['max_samples']
best_max_feat = best_row['max_features']

print(f'Best n_estimators: {best_n_est}')
print(f'Best max_samples: {best_max_samp}')
print(f'Best max_features: {best_max_feat}')
print(f'Best valid accuracy: {best_row["accuracy"]:.5f}')

# Обучаем финальную модель
bagging_best = BaggingClassifier(
    base_estimator=base_svm,
    n_estimators=best_n_est,
    max_samples=best_max_samp,
    max_features=best_max_feat,
    bootstrap=True,
    bootstrap_features=False,
    n_jobs=-1,
    random_state=21
)
bagging_best.fit(X_train, y_train)

# Оценка на тесте
y_pred_test = bagging_best.predict(X_test)

print(f'\naccuracy is {accuracy_score(y_test, y_pred_test):.5f}')
print(f'precision is {precision_score(y_test, y_pred_test, average="weighted"):.5f}')
print(f'recall is {recall_score(y_test, y_pred_test, average="weighted"):.5f}')

Best n_estimators: 30
Best max_samples: 1.0
Best max_features: 1.0
Best valid accuracy: 0.88889

accuracy is 0.87278
precision is 0.87840
recall is 0.87278


## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [10]:
# Базовые модели (те же, что использовались ранее)
base_models = [
    ('svm', SVC(C=10, kernel='rbf', gamma='auto', class_weight=None, 
                probability=True, random_state=21)),
    ('dt', DecisionTreeClassifier(max_depth=21, class_weight='balanced', 
                                   criterion='gini', random_state=21)),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=24, 
                                   class_weight='balanced', criterion='entropy', 
                                   random_state=21))
]

# Перебор параметров
n_splits_list = [2, 3, 4, 5, 6, 7]
passthrough_list = [True, False]

results = []

for n_splits in n_splits_list:
    for passthrough in passthrough_list:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=21)
        
        stacking_clf = StackingClassifier(
            estimators=base_models,
            final_estimator=LogisticRegression(solver='liblinear', random_state=21),
            cv=cv,
            stack_method='predict_proba',
            n_jobs=-1,
            passthrough=passthrough
        )
        stacking_clf.fit(X_train, y_train)
        y_pred = stacking_clf.predict(X_valid)
        
        results.append({
            'n_splits': n_splits,
            'passthrough': passthrough,
            'accuracy': accuracy_score(y_valid, y_pred),
            'precision': precision_score(y_valid, y_pred, average='weighted'),
            'recall': recall_score(y_valid, y_pred, average='weighted')
        })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(['accuracy', 'precision'], ascending=[False, False])
print(results_df.to_string(index=False))

 n_splits  passthrough  accuracy  precision    recall
        4         True  0.911111   0.913269  0.911111
        7         True  0.903704   0.906397  0.903704
        3         True  0.903704   0.906322  0.903704
        2         True  0.903704   0.906190  0.903704
        4        False  0.903704   0.905703  0.903704
        7        False  0.903704   0.905376  0.903704
        6         True  0.903704   0.904500  0.903704
        6        False  0.903704   0.904365  0.903704
        5         True  0.900000   0.902167  0.900000
        5        False  0.900000   0.900558  0.900000
        3        False  0.896296   0.897592  0.896296
        2        False  0.896296   0.896784  0.896296


In [11]:
# Выбираем лучшие параметры
best_row = results_df.iloc[0]
best_n_splits = int(best_row['n_splits'])
best_passthrough = best_row['passthrough']

print(f'Best n_splits: {best_n_splits}')
print(f'Best passthrough: {best_passthrough}')
print(f'Best valid accuracy: {best_row["accuracy"]:.5f}')

# Обучаем финальную модель
cv_best = StratifiedKFold(n_splits=best_n_splits, shuffle=True, random_state=21)

stacking_best = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(solver='liblinear', random_state=21),
    cv=cv_best,
    stack_method='predict_proba',
    n_jobs=-1,
    passthrough=best_passthrough
)
stacking_best.fit(X_train, y_train)

# Оценка на тесте
y_pred_test = stacking_best.predict(X_test)

print(f'\naccuracy is {accuracy_score(y_test, y_pred_test):.5f}')
print(f'precision is {precision_score(y_test, y_pred_test, average="weighted"):.5f}')
print(f'recall is {recall_score(y_test, y_pred_test, average="weighted"):.5f}')

Best n_splits: 4
Best passthrough: True
Best valid accuracy: 0.91111

accuracy is 0.90533
precision is 0.90844
recall is 0.90533


## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [12]:
# Базовые модели
svm = SVC(C=10, kernel='rbf', gamma='auto', class_weight=None, 
          probability=True, random_state=21)
dt = DecisionTreeClassifier(max_depth=21, class_weight='balanced', 
                            criterion='gini', random_state=21)
rf = RandomForestClassifier(n_estimators=100, max_depth=24, 
                            class_weight='balanced', criterion='entropy', 
                            random_state=21)

# Лучшая модель (Voting)
best_model = VotingClassifier(
    estimators=[('svm', svm), ('dt', dt), ('rf', rf)],
    voting='soft',
    weights=[4, 1, 4]
)
best_model.fit(X_train, y_train)

# Предсказания на тесте
y_pred = best_model.predict(X_test)

In [13]:
# анализ ошибок
full_counts = y.value_counts()

df_test = pd.DataFrame({'y_true': y_test, 'y_pred': y_pred})
df_test['is_error'] = df_test['y_true'] != df_test['y_pred']
df_test.index = X_test.index

test_errors = df_test[df_test['is_error']].groupby('y_true').size()
error_rate_weekday = (test_errors / full_counts * 100).fillna(0)

print('Error rate by weekday (% of full dataset class):')
print(error_rate_weekday.sort_values(ascending=False).round(2))
print(f'Worst weekday: {error_rate_weekday.idxmax()} with {error_rate_weekday.max():.2f}% errors\n')

X_test_df = pd.DataFrame(X_test, columns=X.columns)

lab_cols = [c for c in X.columns if c.startswith('labname_')]
error_by_lab = {}
for lab in lab_cols:
    lab_active = X_test_df[lab] == 1
    n_total = lab_active.sum()
    if n_total > 0:
        n_errors = df_test[lab_active]['is_error'].sum()
        error_by_lab[lab] = n_errors / n_total * 100

error_lab_df = pd.Series(error_by_lab).sort_values(ascending=False)
print('Error rate by labname (% of samples with that lab active):')
print(error_lab_df.head(10).round(2))
print(f'Worst labname: {error_lab_df.idxmax()} with {error_lab_df.max():.2f}% errors\n')

user_cols = [c for c in X.columns if c.startswith('uid_user_')]
error_by_user = {}
for user in user_cols:
    user_active = X_test_df[user] == 1
    n_total = user_active.sum()
    if n_total > 0:
        n_errors = df_test[user_active]['is_error'].sum()
        error_by_user[user] = n_errors / n_total * 100

error_user_df = pd.Series(error_by_user).sort_values(ascending=False)
print('Error rate by user (% of samples with that user active):')
print(error_user_df.head(10).round(2))
print(f'Worst user: {error_user_df.idxmax()} with {error_user_df.max():.2f}% errors\n')

Error rate by weekday (% of full dataset class):
0    5.88
5    2.58
4    1.92
6    1.69
1    1.46
2    1.34
3    0.76
dtype: float64
Worst weekday: 0 with 5.88% errors

Error rate by labname (% of samples with that lab active):
labname_lab03       100.00
labname_laba04       25.71
labname_laba04s      24.00
labname_lab05s       16.67
labname_laba06       11.11
labname_code_rvw      7.69
labname_laba06s       6.67
labname_project1      6.45
labname_laba05        0.00
labname_lab03s        0.00
dtype: float64
Worst labname: labname_lab03 with 100.00% errors

Error rate by user (% of samples with that user active):
uid_user_6     50.00
uid_user_17    28.57
uid_user_3     21.43
uid_user_16    20.00
uid_user_18    16.67
uid_user_27    16.67
uid_user_19    15.79
uid_user_2     14.29
uid_user_30    12.50
uid_user_13    11.76
dtype: float64
Worst user: uid_user_6 with 50.00% errors



In [14]:
# сохранение модели
joblib.dump(best_model, '../data/voting_ensembles_model.pkl')
print('Model saved to ../data/voting_ensembles_model.pkl')

print(f'\naccuracy is {accuracy_score(y_test, y_pred):.5f}')
print(f'precision is {precision_score(y_test, y_pred, average="weighted"):.5f}')
print(f'recall is {recall_score(y_test, y_pred, average="weighted"):.5f}')

Model saved to ../data/voting_ensembles_model.pkl

accuracy is 0.90533
precision is 0.90881
recall is 0.90533
